In [ ]:
import pandas as pd
import numpy as np
import os
import warnings

# Silenciar avisos de depreciação
warnings.filterwarnings('ignore')

def gerar_matrizes_retornos_v2(diretorio_data, data_inicio="2010-01-01", data_fim="2025-12-31"):
    print("=== INÍCIO DO MÓDULO DE RETORNOS V2 (ATIVOS + IBOV) ===")
    
    # 1. Carregamento dos Preços Limpos
    caminho_precos = os.path.join(diretorio_data, "lista_economatica_dados_Jan_2010_Dezembro_2025.parquet")
    print("1. A carregar preços dos ativos...")
    df_precos = pd.read_parquet(caminho_precos, index_col='Data', parse_dates=True)
    
    # 2. Carregamento do IBOV
    caminho_ibov = os.path.join(diretorio_data, "IBOV_2010_2026.xlsx")
    print("2. A carregar e alinhar o índice IBOV...")
    df_ibov = pd.read_excel(caminho_ibov)
    df_ibov = df_ibov.rename(columns={'Date': 'Data'}).set_index('Data')
    df_ibov.index = pd.to_datetime(df_ibov.index)
    
    # 3. União e Filtro de Datas
    # Unimos os ativos com o IBOV (join='inner' garante apenas datas comuns)
    df_consolidado = pd.concat([df_precos, df_ibov], axis=1, join='inner')
    
    # Aplicamos o filtro restritivo de 2010 a 2025
    print(f"3. A aplicar filtro temporal: {data_inicio} até {data_fim}")
    df_consolidado = df_consolidado.loc[data_inicio:data_fim]
    
    # 4. Cálculo dos Retornos Simples
    print("4. A calcular Retornos Simples (Discretos)...")
    df_ret_simples = df_consolidado.pct_change().dropna()
    
    # 5. Cálculo dos Retornos Logarítmicos
    print("5. A calcular Retornos Logarítmicos (Contínuos)...")
    df_ret_log = np.log(df_consolidado / df_consolidado.shift(1)).dropna()
    
    # 6. Exportação
    caminho_simples = os.path.join(diretorio_data, "matriz_retornos_simples_v2.csv")
    caminho_log = os.path.join(diretorio_data, "matriz_retornos_logaritmicos_v2.csv")
    
    df_ret_simples.to_csv(caminho_simples)
    df_ret_log.to_csv(caminho_log)
    
    print("\n=== SÍNTESE DA OPERAÇÃO V2 ===")
    print(f"Período: {df_ret_simples.index.min().date()} a {df_ret_simples.index.max().date()}")
    print(f"Total de Ativos + IBOV: {len(df_ret_simples.columns)}")
    print(f"Registros processados: {len(df_ret_simples)} dias")
    print(f"Arquivos salvos em: {diretorio_data}")
    print("===============================================================")
    
    return df_ret_simples, df_ret_log

# ==========================================
# EXECUÇÃO
# ==========================================
pasta_trabalho = r"C:\VSCodeWorkspace\TCC_Final\data"
ret_s, ret_l = gerar_matrizes_retornos_v2(pasta_trabalho)

=== INÍCIO DO MÓDULO DE RETORNOS V2 (ATIVOS + IBOV) ===
1. A carregar preços dos ativos...
2. A carregar e alinhar o índice IBOV...
3. A aplicar filtro temporal: 2010-01-01 até 2025-12-31
4. A calcular Retornos Simples (Discretos)...


TypeError: operation 'truediv' not supported for dtype 'str' with dtype 'str'